# quant-kit — GGUF Quantization Pipeline

> **GitHub**: [DhruvalPtl/quant-kit](https://github.com/DhruvalPtl/quant-kit)  
> **HuggingFace**: [Dhptl](https://huggingface.co/Dhptl)

### Before you start:
1. `Runtime → Change runtime type → GPU (T4)`
2. Add your HF token to Colab Secrets (🔑 key icon, left sidebar):
   - Name: `HF_TOKEN` | Value: your token from https://huggingface.co/settings/tokens

Run cells **top to bottom**. If any cell fails, paste the error in our chat.


In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 1 — Check runtime (GPU / Disk / RAM)
# ═══════════════════════════════════════════════════════════
import subprocess, shutil, psutil

gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader',
                     shell=True, capture_output=True, text=True)
if gpu.returncode == 0:
    print(f'[OK] GPU  : {gpu.stdout.strip()}')
else:
    print('[!!] No GPU! Go to Runtime > Change runtime type > T4 GPU')

disk = shutil.disk_usage('/')
print(f'[OK] Disk : {disk.free/1e9:.1f} GB free of {disk.total/1e9:.1f} GB')

ram = psutil.virtual_memory()
print(f'[OK] RAM  : {ram.available/1e9:.1f} GB available of {ram.total/1e9:.1f} GB')

if disk.free/1e9 < 50:
    print('[!!] Less than 50GB free — may be tight for 12B model')

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 2 — Clone quant-kit & run Linux setup
# ═══════════════════════════════════════════════════════════
import os

REPO    = 'https://github.com/DhruvalPtl/quant-kit.git'
WORKDIR = '/content/quant-kit'

if os.path.exists(WORKDIR):
    print('[OK] quant-kit already cloned — pulling latest...')
    os.system(f'git -C {WORKDIR} pull')
else:
    print('[->] Cloning quant-kit...')
    os.system(f'git clone {REPO} {WORKDIR}')

os.chdir(WORKDIR)
print(f'[OK] Working directory: {os.getcwd()}')
print()

# Run Linux setup: downloads llama.cpp binaries + conversion scripts
# This handles .tar.gz extraction, symlink flattening, and .so versioned links
!python setup_linux.py

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 3 — Verify llama-quantize works (symlink check)
# ═══════════════════════════════════════════════════════════
import os, re, subprocess
from pathlib import Path
from collections import defaultdict

LLAMA_CPP = Path('/content/quant-kit/llama.cpp')

# Auto-fix any missing versioned .so symlinks (3-part: libXXX.so.0.0.NNNN)
three_part = re.compile(r'^(lib.+\.so)\.(\d+)\.\d+\.\d+$')
by_base = defaultdict(list)
for f in LLAMA_CPP.glob('lib*.so.*'):
    m = three_part.match(f.name)
    if m and not f.is_symlink():
        by_base[m.group(1)].append((int(m.group(2)), f.name))

fixed = 0
for base_so, versions in by_base.items():
    versions.sort(reverse=True)
    latest, major = versions[0][1], versions[0][0]
    so_major = LLAMA_CPP / f'{base_so}.{major}'
    if not so_major.exists():
        os.symlink(latest, str(so_major))
        fixed += 1

if fixed:
    print(f'[OK] Created {fixed} missing versioned .so symlinks')

# Test llama-quantize with LD_LIBRARY_PATH
env = {**os.environ, 'LD_LIBRARY_PATH': str(LLAMA_CPP)}
result = subprocess.run([str(LLAMA_CPP / 'llama-quantize'), '--help'],
                        capture_output=True, text=True, env=env)

if result.returncode == 0:
    print('[OK] llama-quantize works!')
    print('[OK] Ready to quantize. Run Cell 4.')
else:
    print(f'[ERR] llama-quantize failed (exit {result.returncode})')
    print(result.stderr[:400])
    print()
    print('Check missing libs:')
    ldd = subprocess.run(['ldd', str(LLAMA_CPP / 'llama-quantize')],
                         capture_output=True, text=True, env=env)
    for l in ldd.stdout.splitlines():
        if 'not found' in l:
            print(f'  MISSING: {l.strip()}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 4 — HuggingFace authentication
# ═══════════════════════════════════════════════════════════
import os
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token

    with open('/content/quant-kit/.env', 'w') as f:
        f.write(f'hf_token = "{hf_token}"\n')

    from huggingface_hub import HfApi
    user = HfApi(token=hf_token).whoami()
    print(f'[OK] Logged in as: {user["name"]}')

except Exception as e:
    print(f'[ERR] {e}')
    print('  1. Click the 🔑 key icon in the left sidebar')
    print('  2. Add secret: Name=HF_TOKEN, Value=your_token')
    print('  3. Toggle "Notebook access" ON')

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 5 — Quantize
# Edit MODEL_ID below before running!
# ═══════════════════════════════════════════════════════════
import os
os.chdir('/content/quant-kit')

MODEL_ID = 'google/gemma-4-12b-it'           # <-- change for different models
QUANTS   = 'Q4_K_M Q5_K_M Q8_0 IQ4_XS'  # <-- quant types to produce

# --delete-src  : frees ~24GB after FP16 conversion (critical on 120GB Colab disk)
# --keep-fp16   : add this flag if you want to keep FP16 for retry without redownload
!python quantize.py \
    --model {MODEL_ID} \
    --quants {QUANTS} \
    --delete-src

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 6 — Benchmark
# ═══════════════════════════════════════════════════════════
import os
os.chdir('/content/quant-kit')

MODEL_NAME = MODEL_ID.split('/')[-1]

# ngl=99: offload all layers to GPU (T4 15GB VRAM — enough for 12B Q4/Q5)
# ngl=0:  CPU only — use if GPU VRAM not enough for the model
!python benchmark.py --model {MODEL_NAME} --ngl 99

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 7 — Generate model card (README)
# ═══════════════════════════════════════════════════════════
import os
os.chdir('/content/quant-kit')

HF_AUTHOR = 'Dhptl'   # <-- your HuggingFace username

!python model_card.py \
    --model {MODEL_NAME} \
    --original {MODEL_ID} \
    --author {HF_AUTHOR}

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 8 — Upload to HuggingFace
# Creates Dhptl/gemma-4-12B-GGUF automatically
# ═══════════════════════════════════════════════════════════
import os
os.chdir('/content/quant-kit')

!python upload.py \
    --model {MODEL_NAME} \
    --author {HF_AUTHOR}

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 9 — Cleanup (run after upload to free disk for next model)
# ═══════════════════════════════════════════════════════════
import shutil, os
from pathlib import Path

model_output = Path(f'/content/quant-kit/output/{MODEL_NAME}')
if model_output.exists():
    shutil.rmtree(str(model_output))
    print(f'[OK] Cleaned: {model_output}')

disk = shutil.disk_usage('/')
print(f'[OK] Disk after cleanup: {disk.free/1e9:.1f} GB free')
print()
print('To quantize another model:')
print('  1. Change MODEL_ID in Cell 5')
print('  2. Re-run Cells 5 → 6 → 7 → 8 → 9')